# 04 — NHANES 2017-2018: Diabetes

**Dataset:** National Health and Nutrition Examination Survey (NHANES), 2017-2018 cycle.
Adults aged 18+ (n ~ 5,852).

**Outcome (`diabetes`):** derived from `DIQ010` ("Has a doctor or health professional ever told
you that you have diabetes?"). Coded 1 if the respondent answered "Yes" or "Borderline", 0 if "No".

**Protected attribute (`RIAGENDR`):** 1 = Male, 2 = Female.

**Covariate blocks for the case-mix waterfall:**
- `demographics`: age, education level, income-to-poverty ratio, race/ethnicity (one-hot)
- `comorbidities`: high blood pressure, high cholesterol, BMI
- `behavioral`: smoking history (100+ cigarettes lifetime), vigorous physical activity
- `access`: health insurance coverage

This notebook runs the shared `run_fairness_analysis` pipeline (see `src/pipeline.py`) on this
dataset, exactly as for datasets 1-3, so results are directly comparable.


In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
RESULTS_DIR = ROOT / "results"
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.datasets import load_nhanes
from src.pipeline import run_fairness_analysis
from src import figures as figs


In [ ]:
d = load_nhanes()
df = d["df"]

print(d["name"])
print("shape:", df.shape)
print("feature columns:", len(d["feature_cols"]))
print()
for val, lbl in d["group_labels"].items():
    sub = df[df[d["group_col"]] == val]
    n_pos = sub[d["target_col"]].sum()
    print(f"{lbl}: n={len(sub)}, positives={n_pos}, prevalence={n_pos / len(sub):.4f}")


## Run the fairness pipeline

We use `threshold="prevalence"`: the diabetes prevalence in this sample (~18%) is far from
50%, so a fixed 0.5 decision threshold would classify almost no one as positive and make
PPV/sensitivity estimates very noisy. Using the training prevalence as the threshold gives
a more balanced operating point, consistent with datasets 2 and 3.


In [ ]:
result = run_fairness_analysis(
    df=df,
    feature_cols=d["feature_cols"],
    target_col=d["target_col"],
    group_col=d["group_col"],
    group_a_value=1.0,
    group_b_value=2.0,
    group_labels=d["group_labels"],
    threshold="prevalence",
    n_boot=1000,
    covariate_blocks=d["covariate_blocks"],
    # Corrected path: preprocessing is fitted on training rows only,
    # and (where a clustering identifier exists) the split, selection
    # CV, calibration folds and bootstrap are patient-grouped.
    cluster_ids=None,
    preprocess_spec=d["preprocess_spec"],
)


## Interpretation

- **Raw gaps:** Men have a slightly higher diabetes prevalence in this sample (19.5% vs
  16.6% in the test set) and a higher PPV (the model is more often right when it flags a
  man as diabetic). The equal-opportunity gap (sensitivity difference, +0.013) and the
  specificity/FPR gap (+0.037) are both small and their 95% bootstrap CIs cross zero --
  unlike datasets 1-3, neither sensitivity nor specificity shows a statistically
  significant sex gap in this sample.
- **Prevalence adjustment:** Holding sensitivity/specificity fixed and recomputing PPV/NPV
  at a common prevalence attenuates the raw PPV gap by ~61% (from +0.070 to +0.027, whose
  CI now crosses zero), consistent with the hypothesis that most of the raw PPV gap is a
  mechanical consequence of the higher male prevalence rather than a difference in how
  well the model discriminates within each group.
- **Only the raw PPV gap is statistically significant** (95% CI [+0.005, +0.137]), and even
  that gap shrinks substantially and loses significance after prevalence adjustment. The
  NPV and predicted-positive-rate gaps are not significant either raw or adjusted (note the
  large negative "attenuation" percentages here simply reflect that these raw gaps are
  already close to zero and noisy, so small absolute changes produce large relative swings).
- **Case-mix waterfall:** Adding comorbidities (blood pressure, cholesterol, BMI) *increases*
  the male coefficient on the diabetes outcome rather than shrinking it -- i.e. controlling
  for these known diabetes risk factors does not explain away the sex gap in this sample;
  if anything the raw gap is partly suppressed by these covariates being correlated with sex.
- **Caveat:** NHANES subgroup sizes here (n~850 per sex in the test set, ~150-170 positives)
  are noticeably smaller than the BRFSS/Diabetes-130 samples, so confidence intervals are
  wide and most point estimates here should be read as suggestive rather than conclusive.

As with the other datasets, none of this should be read as showing the model is "unbiased":
sample size here is simply too small to detect gaps of the magnitude seen in datasets 1-3.
This dataset mainly serves as a smaller, independent replication with much wider
uncertainty, not as evidence against the residual-inequity pattern found elsewhere.


In [ ]:
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

fig1 = figs.plot_calibration_curve(
    result["y_test"], result["prob"], result["g_test"],
    group_labels={1.0: "Male", 2.0: "Female"},
    title="Calibration by sex (calibrated model) -- NHANES 2017-2018 diabetes",
)
fig1.savefig(RESULTS_DIR / "04_nhanes_calibration.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
for key in ("ppv", "npv", "predicted_positive_rate"):
    raw = result["bootstrap_raw"][key]
    adjd = result["bootstrap_prevalence_adjusted"][key]
    gap_dict = {
        "Raw": (raw["point"], raw["ci_low"], raw["ci_high"]),
        "Prevalence-adjusted": (adjd["point"], adjd["ci_low"], adjd["ci_high"]),
    }
    fig = figs.plot_gap_bars(
        gap_dict,
        title=f"{key}: raw vs. prevalence-adjusted gap (Male - Female)",
        ylabel="Gap (Male - Female)",
    )
    fig.savefig(RESULTS_DIR / f"04_nhanes_gap_{key}.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
fig_wf = figs.plot_case_mix_waterfall(
    result["case_mix"],
    title="NHANES 2017-2018: case-mix waterfall (Male vs Female diabetes gap)",
)
fig_wf.savefig(RESULTS_DIR / "04_nhanes_case_mix_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()


## Summary table (for cross-dataset pooling)

In [ ]:
rows = []
for key, boot in result["bootstrap_raw"].items():
    rows.append({
        "dataset": "nhanes_diabetes",
        "metric": key,
        "raw_gap": boot["point"],
        "raw_ci_low": boot["ci_low"],
        "raw_ci_high": boot["ci_high"],
        "prevalence_adjusted_gap": None,
        "prevalence_adjusted_ci_low": None,
        "prevalence_adjusted_ci_high": None,
        "attenuation_pct": None,
    })

for key, boot in result["bootstrap_prevalence_adjusted"].items():
    raw = result["bootstrap_raw"][key]
    attenuation = None
    if raw["point"] != 0:
        attenuation = 100 * (1 - boot["point"] / raw["point"])
    for row in rows:
        if row["metric"] == key:
            row["prevalence_adjusted_gap"] = boot["point"]
            row["prevalence_adjusted_ci_low"] = boot["ci_low"]
            row["prevalence_adjusted_ci_high"] = boot["ci_high"]
            row["attenuation_pct"] = attenuation

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS_DIR / "04_nhanes_summary.csv", index=False)
summary


In [ ]:
# Frozen held-out outputs.
#
# Persist the row-level held-out predictions, the overall and per-subgroup
# discrimination/calibration table, and the fitted preprocessing + estimator +
# calibration objects. Nothing here changes the analysis: every value written
# is read from `result`, which was produced above. These artifacts let any
# later evaluation be answered without refitting a model.
from src import frozen_outputs as fo

frozen_manifest = fo.write_comparison_artifacts(
    result,
    slug="sex_nhanes",
    dataset=d["name"],
    comparison_type="sex",
    comparison="Male vs Female",
    results_dir=RESULTS_DIR,
    cluster_ids=None,
)
print("frozen predictions :", frozen_manifest["predictions"])
print("frozen performance :", frozen_manifest["performance"])
print("serialized objects :", frozen_manifest["model"]["status"])
